In [22]:
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [23]:
# Import dataset
dataset = pd.read_csv(r"C:\Users\Disha\TU Braunschweig\Thesis\Data\Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),BEV (New Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),...,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_BEV (New Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_Available,Log_DC Recharging Speed (km/h),YJ_Purchase price (EUR)
0,Austria,2011,0.18,631.0,308167.0,3.286579,1.6,963,17.500000,175.000000,...,95.000000,30469.93,0.165514,6.448889,6.871091,12.638400,1.455489,1.609438,5.170484,-1.493084
1,Austria,2012,0.13,427.0,316589.4,2.485676,1.0,1063,29.169521,217.394139,...,95.000000,30469.93,0.122218,6.059123,6.969791,12.665364,1.248662,1.609438,5.386301,-1.493084
2,Austria,2013,0.21,656.0,321191.7,2.000156,0.4,1173,48.200000,270.000000,...,142.220000,30469.93,0.190620,6.487684,7.068172,12.679797,1.098664,2.197225,5.602119,-1.493084
3,Austria,2014,0.44,1344.0,330113.5,1.605812,1.0,1393,22.600000,240.000000,...,155.770000,35555.64,0.364643,7.204149,7.239933,12.707195,0.957744,2.639057,5.484797,-0.885657
4,Austria,2015,0.55,1699.0,342083.5,0.896563,0.6,1625,47.700000,381.250000,...,215.220000,42052.00,0.438255,7.438384,7.393878,12.742813,0.640043,3.091042,5.946075,0.277809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Sweden,2020,9.55,27897.0,5012855.0,0.497367,-1.3,47680,39.325301,403.164557,...,286.200000,38311.38,2.356126,10.236310,10.772288,15.427516,0.403708,4.584967,6.001822,-0.451036
374,Sweden,2021,19.09,57454.0,5417760.0,2.163197,1.3,70537,40.904348,493.391304,...,308.730000,40745.01,3.000222,10.958757,11.163907,15.505193,1.151583,5.129899,6.203327,0.003737
375,Sweden,2022,32.84,94603.0,5816415.0,8.369291,3.5,83510,49.314815,594.259259,...,345.440000,43874.27,3.521644,11.457455,11.332734,15.576195,2.237437,5.537334,6.388997,0.697159
376,Sweden,2023,38.42,111338.0,6143187.0,8.548625,1.2,126758,70.307692,649.230769,...,361.794521,45999.05,3.674273,11.620335,11.750043,15.630854,2.256397,5.918894,6.477327,1.244323


In [24]:
# Define the target 
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [25]:
# Load the PCA tools 
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

In [26]:
# Data splitting into Pre and Post COVID
pre_covid_df = dataset[dataset['Year'] < 2020].copy()
post_covid_df = dataset[dataset['Year'] >= 2020].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [27]:
# Check which columns have NaNs and how many
print("Missing Values Per Column (Pre-COVID)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column (Post-COVID)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column (Pre-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           8
dtype: int64

Missing Values Per Column (Post-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64


In [28]:
# Checking which rows are empty
print("Missing Log_CPI in Pre-COVID")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])
print("\n")
print("Missing Log_CPI in Post-COVID")
print(post_covid_df[post_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Pre-COVID
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7   Romania  2016      NaN


Missing Log_CPI in Post-COVID
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [29]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of Pre-COVID data 
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()
    
    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))
    
    return df_clean

# Apply the fix to both datasets 
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [30]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [31]:
# Transform the features into 3 Principal Components using the loaded PCA model 
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [32]:
# Define the targets 
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (243, 3)
Test Shape (PCs): (135, 3)


In [33]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France', 
    'Germany', 'Hungary', 'Italy', 'Netherlands', 'Poland', 
    'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland', 
    'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [34]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Tree Based Models

**1. Random Forest Regressor**

In [35]:
def run_rf_model(X_train, y_train, X_test, y_test, cluster_name):
    # Initialize Random Forest Regressor
    # n_estimators=100 and max_depth=None are standard starting points
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)

    # Back-transform from log scale to real percentage units
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)

    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)

    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - Random Forest")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'rf_model': rf,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [36]:
# Run for both clusters
results_rf_leaders = run_rf_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_rf_followers = run_rf_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - Random Forest
Train R² (Log): 0.9274, R² (Real): 0.7833, MAE (Real): 0.19%
Test  R² (Log): -0.4220, R² (Real): -0.3066, MAE (Real): 8.80%

Followers - Random Forest
Train R² (Log): 0.8980, R² (Real): 0.8513, MAE (Real): 0.12%
Test  R² (Log): -0.7197, R² (Real): -0.3661, MAE (Real): 4.88%



In [37]:
# Combine predictions and actual values
combined_preds = np.concatenate([
    results_rf_leaders['pred_test_real'],
    results_rf_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_rf_leaders['y_test_real'],
    results_rf_followers['y_test_real']
])

# Combine metadata in the same order 
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])
    
# Build final results table
all_rf_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error calculation
all_rf_results['Error_Abs'] = (all_rf_results['Actual_BEV_%'] - all_rf_results['Predicted_BEV_%']).abs()

# Print results by year and cluster
for year in sorted(all_rf_results['Year'].unique()):
    print(f"\n---> RANDOM FOREST: {int(year)} PREDICTION RESULTS")

    year_data = all_rf_results[all_rf_results['Year'] == year]

    # Leaders Display
    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Display
    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))


---> RANDOM FOREST: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         2.185797  19.244203
        Sweden         10.55         1.834759   8.715241
       Denmark          8.16         1.748260   6.411740
        France          7.72         1.170069   6.549931
       Germany          7.56         1.856865   5.703135
       Austria          7.41         1.541302   5.868698
       Finland          5.40         1.469421   3.930579
       Belgium          4.48         1.678723   2.801277
         Italy          3.35         1.770544   1.579456
       Hungary          3.32         1.721795   1.598205
       Romania          3.25         1.469198   1.780802
         Spain          3.14         1.753435   1.386565
Czech Republic          2.61         1.725292   0.884708
        Poland          1.86         1.872462   0.012462

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  Pred

In [38]:
# Save the results in an excel file

# Filter the results as per the clusters
all_rf_results['Cluster'] = all_rf_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_rf_results[all_rf_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_rf_results[all_rf_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial1_RF_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_RF_Followers_Results.xlsx", index=False)

**2. XGBoost Regressor**

In [39]:
def run_xgb_model(X_train, y_train, X_test, y_test, cluster_name):
    # Initialize XGBoost Regressor
    # early_stopping_rounds: stops training if validation score doesn't improve for 10 rounds
    xgb = XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.05, 
        max_depth=5, 
        random_state=42,
        early_stopping_rounds=10
    )

    # We need a validation set for early stopping to monitor performance
    # Using 10% of training data for validation
    from sklearn.model_selection import train_test_split
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=42
    )

    # Fit the model with eval_set for early stopping
    xgb.fit(
        X_train_sub, y_train_sub,
        eval_set=[(X_val, y_val)],
        verbose=False 
    )

    train_pred = xgb.predict(X_train)
    test_pred = xgb.predict(X_test)

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)
    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)
    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - XGBoost (Early Stopping at round {xgb.best_iteration})")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'xgb_model': xgb,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [40]:
# Run for both clusters
results_xgb_leaders = run_xgb_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_xgb_followers = run_xgb_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - XGBoost (Early Stopping at round 17)
Train R² (Log): 0.6346, R² (Real): 0.3228, MAE (Real): 0.39%
Test  R² (Log): -2.9160, R² (Real): -0.9917, MAE (Real): 11.26%

Followers - XGBoost (Early Stopping at round 13)
Train R² (Log): 0.5186, R² (Real): 0.4076, MAE (Real): 0.27%
Test  R² (Log): -2.2342, R² (Real): -0.6664, MAE (Real): 5.87%



In [41]:
# Combine predictions and actual values
combined_preds = np.concatenate([
    results_xgb_leaders['pred_test_real'],
    results_xgb_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_xgb_leaders['y_test_real'],
    results_xgb_followers['y_test_real']
])

# Combine metadata in the same order 
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])
    
# Build final results table
all_xgb_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error calculation
all_xgb_results['Error_Abs'] = (all_xgb_results['Actual_BEV_%'] - all_xgb_results['Predicted_BEV_%']).abs()

# Print results by year and cluster
for year in sorted(all_xgb_results['Year'].unique()):
    print(f"\n---> XGBOOST: {int(year)} PREDICTION RESULTS")

    year_data = all_xgb_results[all_xgb_results['Year'] == year]

    # Leaders Display
    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Display
    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))


---> XGBOOST: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         1.663295  19.766705
        Sweden         10.55         1.642276   8.907724
       Denmark          8.16         1.584630   6.575370
        France          7.72         1.341345   6.378655
       Germany          7.56         1.642276   5.917724
       Austria          7.41         1.563347   5.846653
       Finland          5.40         1.468498   3.931502
       Belgium          4.48         1.528986   2.951014
         Italy          3.35         1.584630   1.765370
       Hungary          3.32         1.642276   1.677724
       Romania          3.25         1.528986   1.721014
         Spain          3.14         1.584630   1.555370
Czech Republic          2.61         1.642276   0.967724
        Poland          1.86         1.604911   0.255089

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  Predicted_

In [42]:
# Save the results in an excel file

# Filter the results as per the clusters
all_xgb_results['Cluster'] = all_xgb_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_xgb_results[all_xgb_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_xgb_results[all_xgb_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial1_XGB_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_XGB_Followers_Results.xlsx", index=False)